# RAMS W01 "One Weak Rack": final 1080p render on Kaggle (free GPUs)

Builds the photoreal aisle, bakes the collapse, renders all 12 shots at **1920x1080** with Cycles
on Kaggle's free **2x T4** GPUs (one Blender per GPU, frames split between them), then makes the
graded, captioned **16:9** and **9:16** MP4s with the temp sound design.

**Settings (right-hand panel):** Accelerator **GPU T4 x2**, Internet **On**.
Then **Save Version > Save & Run All (Commit)**. It runs in the background (up to 12 h).

**Resuming:** if a run stops before every frame is done, open the notebook again, **Add Input >
Your Work >** this notebook's latest version, and **Save & Run All** again. Finished frames are
reused; only the rest is rendered.

Results: **Output** tab > `final/`.

In [ ]:
# ---- settings ---------------------------------------------------------------------------------
REPO = "https://github.com/Yashism/Blender-CC.git"
BRANCH = "claude/gifted-wright-yl8rt3"
RES = "1920x1080"
SAMPLES = 64          # 48 is ~25% faster with slightly softer denoising
MAX_HOURS = 10.5      # stop starting new shots in time to finish and save
BLENDER_URL = "https://download.blender.org/release/Blender4.5/blender-4.5.14-linux-x64.tar.xz"
SHOTS = ["s01_impact:set", "s00_establish:set", "s04m_lapse:set", "s06b_wide:collapse", "s02_dent:set", "s05_last:set", "s06a_fold:collapse",
         "s03_reveal:set", "s09_tag:set", "s06c_rain:collapse", "s07_freeze:collapse", "s04_lapse:set",
         "s09c_pull:set", "s09b_twin:set"]

In [ ]:
# ---- setup: GPUs, Blender 4.5, project ----------------------------------------------------------
import glob, os, shutil, subprocess, time
WORK = "/kaggle/working"
TMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
PROJ = f"{TMP}/blender-cc"
FRAMES = f"{WORK}/frames"
FINAL = f"{WORK}/final"
LOGS = f"{WORK}/logs"
for d in (FRAMES, FINAL, LOGS):
    os.makedirs(d, exist_ok=True)
gpus = [l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines() if l.strip()]
print("\n".join(gpus) or "NO GPU: set Accelerator to GPU T4 x2")
assert gpus, "no GPU"
BL = f"{TMP}/blender/blender"
if not os.path.exists(BL):
    subprocess.run(f"wget -q -O {TMP}/bl.tar.xz {BLENDER_URL} && mkdir -p {TMP}/blender && "
                   f"tar -xf {TMP}/bl.tar.xz -C {TMP}/blender --strip-components=1 && rm {TMP}/bl.tar.xz",
                   shell=True, check=True)
subprocess.run("apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libxkbcommon0 libsm6 ffmpeg > /dev/null 2>&1", shell=True)
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
if not os.path.isdir(PROJ):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, PROJ], check=True)
# reuse frames from a previous run (Add Input > this notebook's output)
for prev in glob.glob("/kaggle/input/*/frames"):
    subprocess.run(f"cp -rn {prev}/. {FRAMES}/", shell=True)
# the shot scripts write to renders/w01/<shot>/frames: point those at /kaggle/working/frames/<shot>
for spec in SHOTS:
    shot = spec.split(":")[0]
    os.makedirs(f"{FRAMES}/{shot}", exist_ok=True)
    os.makedirs(f"{PROJ}/renders/w01/{shot}", exist_ok=True)
    link = f"{PROJ}/renders/w01/{shot}/frames"
    if not os.path.islink(link):
        shutil.rmtree(link, ignore_errors=True)
        os.symlink(f"{FRAMES}/{shot}", link)
print(subprocess.run([BL, "--version"], capture_output=True, text=True).stdout.splitlines()[0])

In [ ]:
# ---- build the set and bake the collapse (about 3 minutes) -------------------------------------
os.chdir(PROJ)
subprocess.run([BL, "-b", "--factory-startup", "-P", "scripts/w01/build_set.py", "--",
                "--save", "blender/w01_set.blend"], check=True, stdout=open(f"{LOGS}/build.log", "w"), stderr=subprocess.STDOUT)
subprocess.run([BL, "-b", "blender/w01_set.blend", "-P", "scripts/w01/shots.py", "--", "--shot", "collapse_bake"],
               stdout=open(f"{LOGS}/collapse.log", "w"), stderr=subprocess.STDOUT)
print([l.strip() for l in open(f"{LOGS}/collapse.log") if l.startswith("[shot]")])

In [ ]:
# ---- render every shot, one Blender per GPU, frames interleaved ---------------------------------
t0 = time.time()
ng = len(gpus)
for spec in SHOTS:
    shot, blend = spec.split(":")
    if (time.time() - t0) / 3600 > MAX_HOURS:
        print("time budget reached; run again to continue")
        break
    spp = 24 if shot == "s09b_twin" else SAMPLES
    procs = []
    for i in range(ng):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
        cmd = [BL, "-b", f"blender/w01_{blend}.blend", "-P", "scripts/w01/shots.py", "--", "--shot", shot,
               "--res", RES, "--samples", str(spp), "--device", "gpu", "--slice", f"{i}/{ng}"]
        procs.append(subprocess.Popen(cmd, stdout=open(f"{LOGS}/{shot}_gpu{i}.log", "a"), stderr=subprocess.STDOUT, env=env))
        time.sleep(20)
    for p in procs:
        p.wait()
    n = len(glob.glob(f"{FRAMES}/{shot}/f_*.png"))
    print(f"{(time.time() - t0) / 3600:5.2f} h  {shot}: {n} frames", flush=True)

In [ ]:
# ---- finish: temp audio, grade + copy + UI, 16:9 and 9:16 MP4s -------------------------------
subprocess.run(["pip", "install", "-q", "numpy", "pillow"])
subprocess.run(["python3", "scripts/w01/sfx.py"], check=True)
for spec in SHOTS:
    subprocess.run(["python3", "scripts/w01/encode.py", spec.split(":")[0]])
r = subprocess.run(["python3", "scripts/w01/post.py", "--src", RES, "--jobs", "4"])
print("post exit", r.returncode)
for p in glob.glob("renders/w01/*.mp4") + glob.glob("renders/w01/clips/*.mp4"):
    shutil.copy(p, FINAL)
for p in sorted(glob.glob(FINAL + "/*")):
    print(f"  {os.path.getsize(p) / 1e6:8.1f} MB  {p}")